*0.2 Math / ML basics*

# Vectors

**The situation.** Your help centre has 4,000 articles and a search box. A customer types *"my card was charged twice"*. Keyword search finds nothing — the article is titled *"Duplicate payment on your statement"*. Not one word in common. The customer opens a ticket instead, and 30% of tickets are like this.

**What a vector is.** A *vector* is a list of numbers. An *embedding model* turns a piece of text into a vector — typically 768 or 1,536 numbers — chosen so that texts with similar meaning get similar numbers. "Charged twice" and "duplicate payment" have no words in common, but their vectors sit close together. Every search, recommendation and RAG system in this repo starts here.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Turn three sentences into vectors.** One API call; the model returns one vector per input. Store them as a NumPy array of 32-bit floats — the format every vector database expects.

In [2]:
import numpy as np
from openai import OpenAI

client = OpenAI(timeout=30)
texts = [
    "my card was charged twice",
    "Duplicate payment on your statement",
    "How to change your profile photo",
]
response = client.embeddings.create(model="text-embedding-3-small", input=texts)

vectors = []
for item in response.data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)  # one row per text

print("shape:", vectors.shape, "→", vectors.shape[0], "texts,", vectors.shape[1], "numbers each")
print("first 5 numbers of the first text:", np.round(vectors[0, :5], 4))
print("memory for 4,000 articles:", round(4_000 * vectors.shape[1] * 4 / 1e6, 1), "MB")
assert vectors.shape == (3, 1536)

shape: (3, 1536) → 3 texts, 1536 numbers each
first 5 numbers of the first text: [ 0.0223 -0.0583  0.019  -0.0455 -0.0342]
memory for 4,000 articles: 24.6 MB


**Reading the output.** Three rows of 1,536 numbers. The numbers themselves mean nothing to a person — they are only useful compared with each other, which the next three items do. 4,000 articles fit in about 25 MB, so the whole help centre sits comfortably in memory.

**Which two are close?** The distance between two vectors, with NumPy. Smaller means more similar.

In [3]:
pairs = [(0, 1), (0, 2), (1, 2)]
distances = {}
for a, b in pairs:
    distances[(a, b)] = float(np.linalg.norm(vectors[a] - vectors[b]))
    print(f"distance({texts[a]!r}, {texts[b]!r}) = {distances[(a, b)]:.3f}")
assert distances[(0, 1)] < distances[(0, 2)]

distance('my card was charged twice', 'Duplicate payment on your statement') = 0.937
distance('my card was charged twice', 'How to change your profile photo') = 1.322
distance('Duplicate payment on your statement', 'How to change your profile photo') = 1.277


**Reading the output.** "Charged twice" is closest to "Duplicate payment" — no shared words, but the smallest distance. The profile-photo article is far from both. That is the whole trick behind semantic search: compare meanings as numbers.

```
"my card was charged twice"          ──▶ [ 0.012, -0.031, 0.044, … ]  ┐ close together
"Duplicate payment on your statement" ──▶ [ 0.015, -0.028, 0.041, … ]  ┘
"How to change your profile photo"    ──▶ [-0.052,  0.019, 0.003, … ]    far away
```

**The rule to remember.** Text → vector → compare. Once text is a vector, "similar meaning" becomes "small distance", which a computer can rank in microseconds.

| Use it when | Don't when | Instead use |
|---|---|---|
| search, recommendations, clustering, RAG — anything that needs "similar meaning" | exact matches: ids, codes, part numbers | keyword search (BM25) or a plain database lookup |

**Watch out**
- Vectors from different models cannot be compared. One model per index, forever; re-embed everything when you change it.
- Always `float32`; `float64` doubles memory for no gain in ranking.
- The vector captures meaning, not facts: "the payment succeeded" and "the payment failed" sit surprisingly close.